### LAB 8- Implementation and Performance Evaluation of Categorical Naive Bayes Classifier 

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [4]:
# Loading the dataset
df = pd.read_csv("Lab8_CSV.csv")
# Displaying first five rows
df.head()

,No,Outlook,Temperature,Humidity,Wind,Play Tennis
0,1,Sunny,Hot,High,Weak,No
1,2,Sunny,Hot,High,Strong,No
2,3,Overcast,Hot,High,Weak,Yes
3,4,Rain,Mild,High,Weak,Yes
4,5,Rain,Cool,Normal,Weak,Yes


In [5]:
# Remove serial number column
df = df.drop("No", axis=1)
print("Dataset:")
display(df)

Dataset:


,Outlook,Temperature,Humidity,Wind,Play Tennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [6]:
# Encode all categorical columns

encoders = {}
for column in df.columns:
    le = LabelEncoder()
    df[column] = le.fit_transform(df[column])
    encoders[column] = le

print("Encoded Dataset:")
display(df)

Encoded Dataset:


,Outlook,Temperature,Humidity,Wind,Play Tennis
0,2,1,0,1,0
1,2,1,0,0,0
2,0,1,0,1,1
3,1,2,0,1,1
4,1,0,1,1,1
5,1,0,1,0,0
6,0,0,1,0,1
7,2,2,0,1,0
8,2,0,1,1,1
9,1,2,1,1,1


In [7]:
# Features

X = df.drop("Play Tennis", axis=1)

# Target

y = df["Play Tennis"]

print("Features:")
display(X.head())

print("Target:")
display(y.head())

Features:


,Outlook,Temperature,Humidity,Wind
0,2,1,0,1
1,2,1,0,0
2,0,1,0,1
3,1,2,0,1
4,1,0,1,1


Target:


0    0
1    0
2    1
3    1
4    1
Name: Play Tennis, dtype: int64

In [8]:
# Split dataset into training and testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

Training Samples : 40
Testing Samples  : 10


In [9]:
# Create model

nb_model = CategoricalNB()

# Train

nb_model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None
,min_categories,None


In [10]:
# Predict test data

y_pred = nb_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Categorical Naive Bayes Accuracy")
print("Accuracy :", accuracy)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Categorical Naive Bayes Accuracy
Accuracy : 0.8

Confusion Matrix
[[1 1]
 [1 7]]

Classification Report
              precision    recall  f1-score   support

           0       0.50      0.50      0.50         2
           1       0.88      0.88      0.88         8

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10



In [11]:
# Encode sample using saved encoders

sample = pd.DataFrame({
    "Outlook":["Sunny"],
    "Temperature":["Cool"],
    "Humidity":["High"],
    "Wind":["Strong"]
})

for column in sample.columns:
    sample[column] = encoders[column].transform(sample[column])

print("Encoded Sample")
display(sample)

Encoded Sample


,Outlook,Temperature,Humidity,Wind
0,2,0,0,0


In [12]:
prediction = nb_model.predict(sample)

probability = nb_model.predict_proba(sample)

label = encoders["Play Tennis"].inverse_transform(prediction)

print("Predicted Class :", label[0])

print("\nClass Probabilities")

classes = encoders["Play Tennis"].classes_

for c,p in zip(classes, probability[0]):
    print(f"{c} : {p:.4f}")

Predicted Class : No

Class Probabilities
No : 0.9256
Yes : 0.0744


### Decision Tree Classifier

In [13]:
dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_pred)

print("Decision Tree Accuracy :", dt_accuracy)

Decision Tree Accuracy : 0.8


In [14]:
dt_sample = dt_model.predict(sample)
dt_prob = dt_model.predict_proba(sample)

dt_label = encoders["Play Tennis"].inverse_transform(dt_sample)

print("Prediction :", dt_label[0])

print("\nProbabilities")

for c,p in zip(classes, dt_prob[0]):
    print(f"{c} : {p:.4f}")

Prediction : No

Probabilities
No : 1.0000
Yes : 0.0000


### Logistic Regression Classifier

In [15]:
lr_model = LogisticRegression(max_iter=500)

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, lr_pred)

print("Logistic Regression Accuracy :", lr_accuracy)

Logistic Regression Accuracy : 0.4


In [16]:
lr_sample = lr_model.predict(sample)
lr_prob = lr_model.predict_proba(sample)

lr_label = encoders["Play Tennis"].inverse_transform(lr_sample)

print("Prediction :", lr_label[0])

print("\nProbabilities")

for c,p in zip(classes, lr_prob[0]):
    print(f"{c} : {p:.4f}")

Prediction : No

Probabilities
No : 0.9439
Yes : 0.0561


### SVM

In [17]:
svm_model = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

svm_model.fit(X_train, y_train)

svm_pred = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, svm_pred)

print("SVM Accuracy :", svm_accuracy)

SVM Accuracy : 0.7


In [18]:
svm_sample = svm_model.predict(sample)
svm_prob = svm_model.predict_proba(sample)

svm_label = encoders["Play Tennis"].inverse_transform(svm_sample)

print("Prediction :", svm_label[0])

print("\nProbabilities")

for c,p in zip(classes, svm_prob[0]):
    print(f"{c} : {p:.4f}")

Prediction : No

Probabilities
No : 0.9751
Yes : 0.0249


### Compare all three models

In [19]:
comparison = pd.DataFrame({
    "Model":[
        "Categorical Naive Bayes",
        "Decision Tree",
        "Logistic Regression",
        "Support Vector Machine"
    ],
    "Accuracy":[
        accuracy,
        dt_accuracy,
        lr_accuracy,
        svm_accuracy
    ],
    "Prediction":[
        label[0],
        dt_label[0],
        lr_label[0],
        svm_label[0]
    ]
})

comparison

,Model,Accuracy,Prediction
0,Categorical Naive Bayes,0.8,No
1,Decision Tree,0.8,No
2,Logistic Regression,0.4,No
3,Support Vector Machine,0.7,No


In [20]:
probability_table = pd.DataFrame({
    "Class": classes,
    "Naive Bayes": probability[0],
    "Decision Tree": dt_prob[0],
    "Logistic Regression": lr_prob[0],
    "SVM": svm_prob[0]
})

probability_table

,Class,Naive Bayes,Decision Tree,Logistic Regression,SVM
0,No,0.925602,1.0,0.943927,0.975141
1,Yes,0.074398,0.0,0.056073,0.024859


### Conclusion


In [21]:
print("Conclusion")
print("1. Data was encoded using LabelEncoder.")
print("2. Dataset was split into 80% training and 20% testing.")
print("3. Categorical Naive Bayes model was trained and evaluated.")
print("4. Decision Tree, Logistic Regression, and SVM were also trained.")
print("5. Model accuracies and prediction probabilities were compared.")

Conclusion
1. Data was encoded using LabelEncoder.
2. Dataset was split into 80% training and 20% testing.
3. Categorical Naive Bayes model was trained and evaluated.
4. Decision Tree, Logistic Regression, and SVM were also trained.
5. Model accuracies and prediction probabilities were compared.


### Analysis Report

The models produced different predictions and probability scores because each uses a different learning approach. **Categorical Naive Bayes** assumes that all features are conditionally independent, so its probabilities are computed from feature frequencies, which may not capture relationships between weather attributes. **Decision Tree** predicts based on a series of learned decision rules, often producing highly confident probabilities because each prediction comes from the majority class in a leaf node. **Logistic Regression** models a linear relationship between the encoded features and the target, while **SVM** finds the optimal decision boundary between classes, leading to different confidence scores and, in some cases, different predictions for the same test instance.
